# 01 Data Collection

Collect weekly Google Trends data and clean monthly Census retail sales data for the Search Lag Analyzer project.

Run this notebook from either the project root or the `notebooks` folder.

In [ ]:
from pathlib import Path

import pandas as pd


def find_project_root() -> Path:
    """Return the project root whether the notebook is run from root or notebooks/."""
    cwd = Path.cwd()
    return cwd.parent if cwd.name == "notebooks" else cwd


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

In [ ]:
import time

from pytrends.request import TrendReq

trends_path = RAW_DIR / "google_trends_weekly.csv"

if trends_path.exists():
    print("Using existing Google Trends file:", trends_path)
    trends_df = pd.read_csv(trends_path, index_col=0, parse_dates=True)
else:
    pytrends = TrendReq(hl="en-US", tz=360)

    # Five categories - one search term each
    searches = {
        "running_shoes": "running shoes",
        "furniture": "buy furniture",
        "laptops": "buy laptop",
        "apparel": "buy clothes",
        "washing_machines": "buy washing machine",
    }

    all_trends = {}

    for category, keyword in searches.items():
        print(f"Pulling: {keyword}")
        pytrends.build_payload(
            [keyword],
            timeframe="2018-01-01 2023-12-31",
            geo="US",
        )
        df = pytrends.interest_over_time()
        if not df.empty:
            all_trends[category] = df[keyword]
        time.sleep(3)  # mandatory pause - Google will block you without this

    trends_df = pd.DataFrame(all_trends)
    trends_df.index = pd.to_datetime(trends_df.index)
    trends_df.to_csv(trends_path)
    print("Saved:", trends_path)

print("Done. Shape:", trends_df.shape)
print(trends_df.head())

## Census Retail Sales

Download the Census retail sales workbook from:

https://www.census.gov/retail/mrts/www/mrtssales92-present.xlsx

Save it as `data/raw/census_retail.xlsx`, then run the cells below.

In [ ]:
census_path = RAW_DIR / "census_retail.xlsx"

if not census_path.exists():
    raise FileNotFoundError(
        f"Download the Census workbook and save it to {census_path} before running this cell."
    )


def load_census_monthly_sales(path: Path) -> pd.DataFrame:
    """Load Census retail sales whether the workbook is one sheet or year-by-year sheets."""
    xls = pd.ExcelFile(path)
    print("Workbook sheets:", xls.sheet_names[:10], "...")

    if "Monthly Sales" in xls.sheet_names:
        return pd.read_excel(
            path,
            sheet_name="Monthly Sales",
            skiprows=4,
            header=0,
        )

    yearly_frames = []
    target_years = [str(year) for year in range(2018, 2024) if str(year) in xls.sheet_names]
    if not target_years:
        raise ValueError("Could not find 2018-2023 sheets in the Census workbook.")

    for sheet_name in target_years:
        sheet = pd.read_excel(path, sheet_name=sheet_name, header=None)
        month_labels = sheet.iloc[4, 2:14].tolist()
        section_labels = sheet.iloc[:, 1].astype(str).str.strip()

        not_adjusted_rows = section_labels[section_labels == "NOT ADJUSTED"].index
        if len(not_adjusted_rows) == 0:
            raise ValueError(f"Could not find NOT ADJUSTED section in sheet {sheet_name}.")

        adjusted_rows = section_labels[section_labels == "ADJUSTED"].index
        start_row = not_adjusted_rows[0] + 1
        end_row = adjusted_rows[0] if len(adjusted_rows) else len(sheet)

        yearly = sheet.iloc[start_row:end_row, [1, *range(2, 14)]].copy()
        yearly.columns = ["category", *month_labels]
        yearly = yearly.dropna(subset=["category"])
        yearly_frames.append(yearly)

    return pd.concat(yearly_frames, ignore_index=True)


census_raw = load_census_monthly_sales(census_path)

print(census_raw.head(10))
print(census_raw.shape)
print(census_raw.columns.tolist()[:10])

In [ ]:
# Print all category names so you can identify the right rows.
category_names = census_raw.iloc[:, 0].dropna().astype(str).str.strip().drop_duplicates().tolist()
print(category_names)

In [ ]:
# Adjust these strings exactly to what you see printed above if Census labels change.
categories_to_keep = [
    "Clothing and clothing accessories stores",
    "Furniture and home furnishings stores",
    "Electronics and appliance stores",
]

category_aliases = {
    "Clothing and clothing accessories stores": [
        "Clothing and clothing accessories stores",
        "Clothing and clothing access. stores",
    ],
    "Furniture and home furnishings stores": [
        "Furniture and home furnishings stores",
    ],
    "Electronics and appliance stores": [
        "Electronics and appliance stores",
    ],
}

alias_to_category = {
    alias: category
    for category, aliases in category_aliases.items()
    for alias in aliases
}

category_col = census_raw.columns[0]
census_raw[category_col] = census_raw[category_col].astype(str).str.strip()

census_filtered = census_raw[
    census_raw[category_col].isin(alias_to_category)
].copy()
census_filtered[category_col] = census_filtered[category_col].map(alias_to_category)

if census_filtered.empty:
    raise ValueError("No matching Census categories found. Review the printed category names above.")

print(census_filtered)

In [ ]:
census_filtered = census_filtered.rename(columns={census_filtered.columns[0]: "category"})

census_long = census_filtered.melt(
    id_vars=["category"],
    var_name="date",
    value_name="sales_millions",
)

census_long["date"] = pd.to_datetime(census_long["date"], errors="coerce")
census_long = census_long.dropna(subset=["date", "sales_millions"])
census_long = census_long[
    (census_long["date"] >= "2018-01-01")
    & (census_long["date"] <= "2023-12-31")
]

census_clean_path = PROCESSED_DIR / "census_clean.csv"
census_long.to_csv(census_clean_path, index=False)
print("Saved:", census_clean_path)
print("Census data ready. Shape:", census_long.shape)
print(census_long.head())